In [ ]:
import torch
import numpy as np
from actor_critic_pact import ActorCritic_PACT, ContextDecoder
from actor_critic_pact_pos import ActorCritic_PACT_Pos
import os

In [ ]:
actor_critic_pos = ActorCritic_PACT_Pos(57,1030,12,[512,256,128],[1024,256,128,64],57*10,16,11,[128,64],'tanh',1.0)
decoder_pos      = ContextDecoder(27,[64,128],69)

actor_critic         = ActorCritic_PACT(57,1030,12,[512,256,128],[1024,256,128,64],57*10,16,11,[128,64],'tanh',1.0)
decoder = ContextDecoder(27,[64,128],69)

In [ ]:
model_path = "/home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/logs/go1_pact_pos_rough/Mar05_14-19-13_pact_pos_100hz/model_3100.pt"
loaded_model = torch.load(model_path,map_location='cpu')

In [ ]:
act_state_dict = loaded_model["model_state_dict"]
dec_state_dict = loaded_model["decoder_state_dict"]

for key in list(act_state_dict.keys()):
    act_state_dict[key.replace("_orig_mod.", "")] = act_state_dict.pop(key)
    
for key in list(dec_state_dict.keys()):
    dec_state_dict[key.replace("_orig_mod.", "")] = dec_state_dict.pop(key)

In [ ]:
actor_critic_pos.load_state_dict(act_state_dict)
decoder_pos.load_state_dict(dec_state_dict)

In [ ]:
print(act_state_dict.keys())

In [ ]:
actor_critic.context_encoder.load_state_dict(actor_critic_pos.context_encoder.state_dict())
actor_critic.act_trunk.load_state_dict(actor_critic_pos.act_trunk.state_dict())
actor_critic.critic.load_state_dict(actor_critic_pos.critic.state_dict())

actor_critic.act_tau_out.load_state_dict(actor_critic_pos.act_tau_out.state_dict())
actor_critic.act_pos_out.load_state_dict(actor_critic_pos.act_pos_out.state_dict())

In [ ]:
decoder.load_state_dict(dec_state_dict)

In [ ]:
temp_state_dict = {
    "model_state_dict":actor_critic.state_dict(),
    "decoder_state_dict":decoder.state_dict()
}

In [ ]:
converted_path = "pretained_checkpoints/rl_pos/go1_pact_pos_rough/Mar05_14-19-13_pact_pos_100hz"

if not os.path.exists(converted_path):
    os.makedirs(converted_path, exist_ok=True)

torch.save(temp_state_dict, os.path.join(converted_path, "model_3100_converted.pt"))